In [2]:
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
import os
import shutil
from sklearn.metrics import accuracy_score
import hls4ml

%matplotlib inline
seed = 0
np.random.seed(seed)

tf.random.set_seed(seed)

os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

model_name = "MNIST_CNN_HGQ_DynamicTraining_model"

In [3]:
print(keras.__version__)  
print(hls4ml.__version__)

3.12.1
1.3.0


In [4]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()
X_test = X_test.astype("float32") / 255
y_test = keras.utils.to_categorical(y_test, 10)

In [5]:
from keras.models import load_model
import hgq.layers


model = load_model(f"{model_name}.keras")
score = model.evaluate(X_test, y_test)

I0000 00:00:1777878161.406453    4632 service.cc:152] XLA service 0x7199a8003be0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1777878161.406500    4632 service.cc:160]   StreamExecutor device (0): Host, Default Version
2026-05-04 07:02:41.499964: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


 80/313 ━━━━━━━━━━━━━━━━━━━━ 3s 16ms/step - accuracy: 0.9099 - loss: 0.2823 

I0000 00:00:1777878161.822869    4632 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9282 - loss: 0.2399


In [6]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 28, 28, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d (QConv2D)              │ (None, 26, 26, 16)     │           726 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 13, 13, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_conv2d_1 (QConv2D)            │ (None, 11, 11, 32)     │        19,186 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ q_dense (QDense)                │ (None, 10)             │        32,045 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 129,767 (468.68 KB)

 Trainable params: 38,904 (151.97 KB)

 Non-trainable params: 13,053 (12.76 KB)

 Optimizer params: 77,810 (303.95 KB)

In [7]:
import hls4ml

output_dir = f"hls4ml_prj_{model_name}"

hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')

#hls_config['Model']['Strategy'] = 'Distributed Arithmetic'
#proj_name = f"{str(model_to_test)}_{str(model_revision)}_hls4ml_prj_{str(hls4ml_revision)}"

hls_model = hls4ml.converters.convert_from_keras_model(
    model,    
    backend='vitis',
    hls_config=hls_config,
    #io_type='io_stream',
    #proj_name = proj_name,
    output_dir=output_dir, 
    board       = 'kv260',
    part='xck26-sfvc784-2LV-c',
    clock_period='5',
)
hls_model.compile()

In [8]:
hls_model.build(csim=False, synth=True, cosim=False)


****** vitis-run v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Mon May  4 07:03:38 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
Sourcing Tcl script '/work/development/MNIST/CNN_HGQ_DynamicTraining/hls4ml_prj_MNIST_CNN_HGQ_DynamicTraining_model/build_prj.tcl'
INFO: [HLS 200-1510] Running: open_project myproject_prj 
Resolution: For help on HLS 200-2182 see docs.amd.com/access/sources/dita/topic?Doc_Version=2025.2%20English&url=ug1448-hls-guidance&resourceid=200-2182.html
INFO: [HLS 200-10] Opening solution '/work/development/MNIST/CNN_HGQ_DynamicTraining/hls4ml_prj_MNIST_CNN_HGQ_DynamicTraining_model/myproject_prj'.
INFO: [HLS 200-1510] Running: set_top myproject 
INFO: [HLS 200-1510] Running: add_files firmware/myproject.cpp -cflags -std=c++0x 
INFO: [HLS 200-10] Adding design file 'firmware/myproject.cpp' to 

KeyboardInterrupt: 